# Prueba breve de preentrenamiento del modelo de 50M

Entrena durante unas pocas actualizaciones sobre una porción pequeña de FineWeb-Edu, evalúa, genera texto y guarda un checkpoint local. Funciona con CUDA, Apple MPS o CPU.

In [10]:
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
import tiktoken

PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'src' / 'llm_mini_lab').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró la raíz del proyecto llm-mini-lab')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from llm_mini_lab.models import GPTModel
from llm_mini_lab.training import (
    GPT_CONFIG_50M, create_dataloader_smollm, generate_text_simple,
    text_to_token_ids, token_ids_to_text,
)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('PyTorch:', torch.__version__)
print('Dispositivo:', device)
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

PyTorch: 2.13.0
Dispositivo: mps


In [11]:
SEED = 123
MAX_LENGTH = 128
BATCH_SIZE = 2
MAX_TOKENS = 5_000_000
MAX_UPDATES = MAX_TOKENS // (BATCH_SIZE * MAX_LENGTH)
VAL_MOD = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1

torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

In [12]:
train_loader, val_loader = create_dataloader_smollm(
    batch_size=BATCH_SIZE, max_length=MAX_LENGTH, val_mod=VAL_MOD,
    seed=SEED, max_tokens=MAX_TOKENS, num_workers=0,
    config_name='cosmopedia-v2', show_progress=True,
    use_rows_api=True, rows_page_size=100,
)

xb, yb = next(iter(train_loader))
# Para este smoke test evitamos recorrer de nuevo SmolLM Corpus para validación.
# La validación real del entrenamiento completo debe usar val_loader.
val_eval_loader = [(xb.clone(), yb.clone())]
assert xb.shape == yb.shape == (BATCH_SIZE, MAX_LENGTH)
assert torch.equal(yb[:, :-1], xb[:, 1:])
print('Micro-batch:', tuple(xb.shape))
print('Tokens máximos del flujo:', f'{MAX_TOKENS:,}')

SmolLM: descargando/tokenizando:   0%|          | 728/5.00M [00:00<1:36:42, 862tok/s]

Micro-batch: (2, 128)
Tokens máximos del flujo: 5,000,000


In [13]:
cfg = {**GPT_CONFIG_50M, 'context_length': MAX_LENGTH}
model = GPTModel(cfg)
model.out_head.weight = model.tok_emb.weight
model = model.to(device)

n_params = sum(parameter.numel() for parameter in model.parameters())
assert n_params <= 50_000_000, f'El modelo tiene {n_params:,} parámetros'
print(f'Parámetros entrenables: {n_params:,} ({n_params / 1e6:.2f}M)')

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

Parámetros entrenables: 47,854,080 (47.85M)


In [ ]:
model.train()
train_losses = []
tokens_seen = 0

for update, (inputs, targets) in enumerate(train_loader, start=1):
    inputs = inputs.to(device)
    targets = targets.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
    if not torch.isfinite(loss):
        raise FloatingPointError(f'Pérdida no finita en la actualización {update}')
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    train_losses.append(loss.item())
    tokens_seen += inputs.numel()
    print(f'Actualización {update:02d}/{MAX_UPDATES} | loss {loss.item():.4f}')
    if update >= MAX_UPDATES:
        break

assert len(train_losses) == MAX_UPDATES, 'El stream terminó antes de completar la prueba'
print(f'Entrenamiento de prueba completado: {tokens_seen:,} tokens')

SmolLM: descargando/tokenizando:   0%|          | 1.58k/5.00M [00:01<1:00:49, 1.37ktok/s]

Actualización 01/19531 | loss 333.2481
Actualización 02/19531 | loss 290.6427
Actualización 03/19531 | loss 200.4537
Actualización 04/19531 | loss 134.9523


SmolLM: descargando/tokenizando:   0%|          | 2.52k/5.00M [00:02<1:05:17, 1.28ktok/s]

Actualización 05/19531 | loss 92.0385
Actualización 06/19531 | loss 79.7433
Actualización 07/19531 | loss 73.1981
Actualización 08/19531 | loss 71.1564


SmolLM: descargando/tokenizando:   0%|          | 3.20k/5.00M [00:02<1:07:41, 1.23ktok/s]

...

Actualización 2090/19531 | loss 8.1163
Actualización 2091/19531 | loss 7.8362
Actualización 2092/19531 | loss 9.4800
Actualización 2093/19531 | loss 9.6731


SmolLM: descargando/tokenizando:  11%|█         | 537k/5.00M [08:46<1:24:44, 878tok/s]  

Actualización 2094/19531 | loss 10.1694
Actualización 2095/19531 | loss 8.9689
Actualización 2096/19531 | loss 7.7231


SmolLM: descargando/tokenizando:  11%|█         | 538k/5.00M [08:47<1:22:49, 898tok/s]

Actualización 2097/19531 | loss 8.7491
Actualización 2098/19531 | loss 7.6632


In [ ]:
model.eval()
val_losses = []
with torch.inference_mode():
    for inputs, targets in val_eval_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        logits = model(inputs)
        val_loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        val_losses.append(val_loss.item())

mean_val_loss = sum(val_losses) / len(val_losses)
print(f'Loss inicial: {train_losses[0]:.4f}')
print(f'Loss final: {train_losses[-1]:.4f}')
print(f'Loss de validación: {mean_val_loss:.4f}')

In [ ]:
tokenizer = tiktoken.get_encoding('gpt2')
prompt = text_to_token_ids('Artificial intelligence', tokenizer).to(device)
generated_ids = generate_text_simple(
    model, prompt, max_new_tokens=20, context_size=MAX_LENGTH
)
print('Muestra:', token_ids_to_text(generated_ids.cpu(), tokenizer))

checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'gpt-50m-smoke-test.pt'
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': cfg,
    'updates': len(train_losses),
    'tokens_seen': tokens_seen,
}, checkpoint_path)
print('Checkpoint guardado en:', checkpoint_path)